In [3]:
import pandas as pd

In [4]:
df = pd.read_csv('/content/Dataset for Data Analytics - Sheet1.csv')

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [5]:
df

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1195,ORD201195,2024-06-20,C21126,Desk,1,107.04,392 Main St,Credit Card,Cancelled,TRK38009181,6,FREESHIP,Google,107.04
1196,ORD201196,2024-03-04,C20095,Monitor,2,662.53,778 Main St,Online,Cancelled,TRK69207593,5,NaN,Facebook,1325.06
1197,ORD201197,2023-07-13,C79674,Tablet,2,436.84,275 Main St,Online,Delivered,TRK88039356,2,FREESHIP,Instagram,873.68
1198,ORD201198,2024-08-22,C64753,Chair,4,262.52,509 Main St,Debit Card,Cancelled,TRK71683331,4,WINTER15,Instagram,1050.08


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [6]:
# Install required libraries in Colab
!pip install pandera scikit-learn

import numpy as np
from sklearn.impute import KNNImputer
import pandera as pa


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [7]:
#The Missing Data Decision Matrix

def handle_missing_data(data):
    df_clean = data.copy()

    # Calculate missingness proportion per feature
    missing_props = df_clean.isnull().mean()

    for col in df_clean.columns:
        prop = missing_props[col]

        if prop == 0:
            continue

        # Rule 1: < 5% Missing -> Drop Rows
        elif prop < 0.05:
            df_clean = df_clean.dropna(subset=[col])

        # Rule 2: 5% - 20% Missing -> Global Median Imputation (for skewed numeric)
        elif 0.05 <= prop <= 0.20:
            if pd.api.types.is_numeric_dtype(df_clean[col]):
                df_clean[col] = df_clean[col].fillna(df_clean[col].median())
            else:
                # Sub-Group Conditional for categorical (using mode)
                df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

        # Rule 3: > 20% Missing -> Multi-Dimensional Estimation (KNN)
        elif prop > 0.20:
            if pd.api.types.is_numeric_dtype(df_clean[col]):
                knn_imputer = KNNImputer(n_neighbors=5)
                # Note: KNN returns a numpy array, so we assign it back to the column
                df_clean[col] = knn_imputer.fit_transform(df_clean[[col]])

    return df_clean


df = handle_missing_data(df)

In [8]:
#Neutralizing Outliers (IQR Capping)

def clip_outliers_iqr(data, numeric_columns):
    df_clipped = data.copy()

    for col in numeric_columns:
        # Calculate Q1, Q3, and IQR
        Q1 = df_clipped[col].quantile(0.25)
        Q3 = df_clipped[col].quantile(0.75)
        IQR = Q3 - Q1

        # Define statistical boundaries
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        # Apply vectorized clipping via NumPy
        df_clipped[col] = np.clip(df_clipped[col], lower_bound, upper_bound)

    return df_clipped

# Select only numeric columns for outlier treatment
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# Exclude target variable from being clipped if necessary
# num_cols.remove('target_column')

df = clip_outliers_iqr(df, num_cols)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [9]:
#Vectorized Computation & Categorical Translation

# Select categorical columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

# Apply One-Hot Encoding (converts categories into 0/1 indicator columns)
# drop_first=True helps prevent basic multicollinearity
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [10]:
#The Collinearity Eradication Algorithm

def eradicate_collinearity(data, target_col, threshold=0.80):
    df_uncorrelated = data.copy()

    # Step 1: Build Absolute Matrix
    corr_matrix = df_uncorrelated.corr().abs()

    # Step 2: Isolate Upper Triangle
    upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

    # Step 3: Identify Pairs > threshold
    to_drop = []
    for col in upper_tri.columns:
        highly_correlated_with = upper_tri.index[upper_tri[col] > threshold].tolist()

        # Step 4: Target Comparison
        for paired_col in highly_correlated_with:
            # Check which feature has a lower correlation with the target variable
            corr_col = abs(df_uncorrelated[col].corr(df_uncorrelated[target_col]))
            corr_paired = abs(df_uncorrelated[paired_col].corr(df_uncorrelated[target_col]))

            if corr_col < corr_paired:
                to_drop.append(col)
            else:
                to_drop.append(paired_col)

    # Drop the duplicate weakest links
    to_drop = list(set(to_drop))
    df_uncorrelated = df_uncorrelated.drop(columns=to_drop)

    return df_uncorrelated

# Replace 'target_column' with the actual name of what you are trying to predict
# df = eradicate_collinearity(df, 'target_column', threshold=0.80)

In [11]:
#Feature Engineering

# Feature 1: Ratio between two important numeric variables
# df['new_ratio_feature'] = df['col1'] / (df['col2'] + 0.0001) # Added small constant to prevent division by zero

# Feature 2: Polynomial feature (squaring a critical variable to capture non-linear effects)
# df['new_squared_feature'] = np.square(df['col3'])

# Feature 3: Interaction term (multiplying two variables that might have a combined effect)
# df['new_interaction_feature'] = df['col4'] * df['col5']

In [12]:
#Runtime Structural Contracts with Pandera

# Define your strict schema based on your final expected columns and boundaries
schema = pa.DataFrameSchema({
    # Example assertions: Replace with your actual final column names
    # "target_column": pa.Column(float, nullable=False),
    # "new_ratio_feature": pa.Column(float, checks=pa.Check.ge(0)), # Greater than or equal to 0
    # Add other columns as needed
})

# Lazy Validation: collects all structural failures into a single diagnostic report
try:
    schema.validate(df, lazy=True)
    print("Data Integrity Vault Check: PASSED. Pipeline ready for downstream estimators.")
except pa.errors.SchemaErrors as err:
    print("Data Integrity Vault Check: FAILED.")
    print(err.failure_cases)

/usr/local/lib/python3.12/dist-packages/pandera/_pandas_deprecated.py:144: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


Data Integrity Vault Check: PASSED. Pipeline ready for downstream estimators.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [13]:
#Bridging the Gap with Feast (Feature Store)

!pip uninstall -y pyarrow
!pip uninstall -y feast
!pip install feast

# Initialize a Feast repository locally in your Colab environment
!feast init feature_repo
%cd feature_repo

# Feast uses a configuration file (feature_store.yaml) and Python definitions to build the registry.
# Here is an example of what your feature definition (e.g., in a file named 'features.py') would look like:

from datetime import timedelta
from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32, Int64

# 1. Define the Entity (The primary key of your dataset, e.g., a Customer ID or Transaction ID)
# user_entity = Entity(name="user_id", join_keys=["user_id"])

# 2. Define the Data Source (Pointing to your cleaned Parquet file - see Step 9)
# clean_data_source = FileSource(
#     path="/content/cleaned_dataset.parquet",
#     timestamp_field="event_timestamp" # Feast requires a timestamp for Point-in-Time correctness
# )

# 3. Define the Feature View (The actual features you engineered and cleaned)
# engineered_features_view = FeatureView(
#     name="engineered_features",
#     entities=[user_entity],
#     ttl=timedelta(days=1),
#     schema=[
#         Field(name="new_ratio_feature", dtype=Float32),
#         Field(name="new_squared_feature", dtype=Float32),
#     ],
#     online=True,
#     source=clean_data_source,
# )

Found existing installation: pyarrow 25.0.0
Uninstalling pyarrow-25.0.0:
  Successfully uninstalled pyarrow-25.0.0
Found existing installation: feast 0.64.0
Uninstalling feast-0.64.0:
  Successfully uninstalled feast-0.64.0
  Using cached feast-0.64.0-py3-none-any.whl.metadata (27 kB)
  Using cached pyarrow-25.0.0-cp312-cp312-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
Using cached feast-0.64.0-py3-none-any.whl (8.3 MB)
Using cached pyarrow-25.0.0-cp312-cp312-manylinux_2_28_x86_64.whl (50.1 MB)


Traceback (most recent call last):
  File "/usr/local/bin/feast", line 5, in <module>
    from feast.cli.cli import cli
  File "/usr/local/lib/python3.12/dist-packages/feast/__init__.py", line 13, in <module>
    from feast.infra.offline_stores.redshift_source import RedshiftSource
  File "/usr/local/lib/python3.12/dist-packages/feast/infra/offline_stores/redshift_source.py", line 25, in <module>
    @typechecked
     ^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/typeguard/_decorators.py", line 201, in typechecked
    if isfunction(retval):
       ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/inspect.py", line 378, in isfunction
    def isfunction(object):
    
KeyboardInterrupt
^C
/content/feature_repo/feature_repo


In [14]:
#Dual-Store Output (Parquet)

# Save the final, validated dataframe to a Parquet file
# This acts as your Offline Store output for model training
df.to_parquet('/content/cleaned_dataset.parquet', engine='pyarrow', index=False)

print("Enterprise Pipeline Complete: Data secured, processed, validated, and stored.")

Enterprise Pipeline Complete: Data secured, processed, validated, and stored.
